In [ ]:
#importent notebook! here we only used the syntheticly labeled message on the training, the results are very good so 
#migh be worth consider implementing with the linear binary classifier approch, or maybe just add this as a feature to the Sahar system 
import pandas as pd
import numpy as np
from tqdm import tqdm_pandas
from tqdm.notebook import tqdm
from transformers import BertModel, BertTokenizerFast, Trainer, TrainingArguments, AutoModelForSequenceClassification, AutoTokenizer, BertForSequenceClassification, BertTokenizer
import torch
from datasets import Dataset
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, precision_recall_fscore_support,  roc_auc_score, fbeta_score

import warnings
warnings.filterwarnings("ignore")

/home/ronfr/.conda/envs/alephbert_eval/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# hyperparameters
batch_size = 16
epochs = 3
learning_rate = 2e-5

In [3]:
model_name = 'onlplab/alephbert-base'
tokenizer = AutoTokenizer.from_pretrained(model_name)
bert_model = AutoModelForSequenceClassification.from_pretrained(model_name , num_labels=2)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
bert_model = bert_model.to(device)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at onlplab/alephbert-base and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [4]:
messages_path = 'trainDatasets/combined_riskfree_riskfull_messages_syntatic.csv'

messages_df = pd.read_csv(messages_path)

messages_df['engagement_id'] = messages_df['engagement_id'].astype(str)
messages_df = messages_df[messages_df['anonymized'].notna()]
messages_df['name'] = messages_df['name'].fillna('-')

In [5]:
# for better results we take only text from help seeker
messages_df = messages_df[messages_df['seeker'] == True]

messages_df['label'] = messages_df['label'].astype(int)

# split to train and test stratisfied by label
train_df, test_df = train_test_split(messages_df, test_size=0.2, stratify=messages_df['label'], random_state=42)

train_df['message_id'] = train_df['message_id'].astype(str)
test_df['message_id'] = test_df['message_id'].astype(str)

In [6]:
# creating Dataset objects
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

# mapping the text into inputs that fits the model
def tokenize(batch):
    return tokenizer(batch['anonymized'], padding='max_length', truncation=True, max_length=512)

train_dataset = train_dataset.map(tokenize, batched=True, batch_size=16)
test_dataset = test_dataset.map(tokenize, batched=True, batch_size=16)

# setting the format to pytorch tensors
train_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
test_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

Map: 100%|██████████| 81858/81858 [00:25<00:00, 3261.43 examples/s]


***training code without weighting minority class***

In [ ]:
optimizer = torch.optim.AdamW(bert_model.parameters(), lr=learning_rate)
bert_model.train()

#progress_bar = tqdm(range(epochs * len(train_loader)), desc="Training")
total_batches = len(train_loader)
for epoch in range(epochs):
    print(f"\nEpoch {epoch + 1}/{epochs}")
    for i, batch in enumerate(train_loader):
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)

        outputs = bert_model(input_ids, attention_mask=attention_mask, labels=labels)

        loss = outputs.loss
        loss.backward()
        optimizer.step()
        #progress_bar.update(1)
        if (i + 1) % 1000 == 0:
            batches_done = i + 1
            batches_left = total_batches - batches_done
            print(f"  [Epoch {epoch + 1}] Batch {batches_done}/{total_batches} completed. {batches_left} remaining.")

***training code with minority class weighting***

In [8]:
from torch.nn import CrossEntropyLoss
optimizer = torch.optim.AdamW(bert_model.parameters(), lr=learning_rate)
bert_model.train()
class_weights = torch.tensor([1.0, 9.0]).to(device)  # weight inversely proportional to class freq
loss_fn = CrossEntropyLoss(weight=class_weights)

total_batches = len(train_loader)

for epoch in range(epochs):
    print(f"\nEpoch {epoch + 1}/{epochs}")
    for i, batch in enumerate(train_loader):
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)

        # Forward pass — get raw logits
        outputs = bert_model(input_ids, attention_mask=attention_mask)
        logits = outputs.logits

        # Calculate loss manually
        loss = loss_fn(logits, labels)

        loss.backward()
        optimizer.step()

        if (i + 1) % 1000 == 0:
            batches_done = i + 1
            batches_left = total_batches - batches_done
            print(f"  [Epoch {epoch + 1}] Batch {batches_done}/{total_batches} completed. {batches_left} remaining.")


Epoch 1/3
  [Epoch 1] Batch 1000/20465 completed. 19465 remaining.
  [Epoch 1] Batch 2000/20465 completed. 18465 remaining.
  [Epoch 1] Batch 3000/20465 completed. 17465 remaining.
  [Epoch 1] Batch 4000/20465 completed. 16465 remaining.
  [Epoch 1] Batch 5000/20465 completed. 15465 remaining.
  [Epoch 1] Batch 6000/20465 completed. 14465 remaining.
  [Epoch 1] Batch 7000/20465 completed. 13465 remaining.
  [Epoch 1] Batch 8000/20465 completed. 12465 remaining.
  [Epoch 1] Batch 9000/20465 completed. 11465 remaining.
  [Epoch 1] Batch 10000/20465 completed. 10465 remaining.
  [Epoch 1] Batch 11000/20465 completed. 9465 remaining.
  [Epoch 1] Batch 12000/20465 completed. 8465 remaining.
  [Epoch 1] Batch 13000/20465 completed. 7465 remaining.
  [Epoch 1] Batch 14000/20465 completed. 6465 remaining.
  [Epoch 1] Batch 15000/20465 completed. 5465 remaining.
  [Epoch 1] Batch 16000/20465 completed. 4465 remaining.
  [Epoch 1] Batch 17000/20465 completed. 3465 remaining.
  [Epoch 1] Batch 1

***model evaluation***

In [9]:
bert_model.eval()
labels = []
preds = []
pred_probs = []

for batch in test_loader:
    input_ids = batch['input_ids'].to(device)
    attention_mask = batch['attention_mask'].to(device)
    label = batch['label'].to(device)

    with torch.no_grad():
        outputs = bert_model(input_ids, attention_mask=attention_mask)

    logits = outputs.logits
    probabilities = torch.softmax(logits, dim=-1)
    predictions = torch.argmax(logits, dim=-1)

    labels.extend(label.cpu().numpy())
    preds.extend(predictions.cpu().numpy())
    pred_probs.extend(probabilities[:, 1].cpu().numpy())

In [10]:
accuracy = accuracy_score(labels, preds)
precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
roc_auc = roc_auc_score(labels, pred_probs)
f2 = fbeta_score(labels, preds, beta=2)

print(f'Accuracy: {accuracy}')
print(f'Precision: {precision}')
print(f'Recall: {recall}')
print(f'F1: {f1}')
print(f'ROC-AUC: {roc_auc}')
print(f'F2: {f2}')

Accuracy: 0.9306970607637617
Precision: 0.6225790832795352
Recall: 0.8856618069108024
F1: 0.7311756622281192
ROC-AUC: 0.9668764540561712
F2: 0.8166440850199


In [11]:
torch.save(bert_model.state_dict(), 'model_weights_anon_original.pth')

In [12]:
messages = [
"היום היה יום נחמד, יצאתי לטיול בפארק.",
"אני מתכנן לצאת לארוחת ערב עם חברים בסוף השבוע.",
"נהנתי מאוד לראות את הסרט החדש בקולנוע.",
"אני אוהבת לבלות עם המשפחה שלי בסופי שבוע.",
"הלכתי לחדר כושר והרגשתי מעולה אחרי האימון.",
"אני מתרגש לטוס לחופשה בקרוב.",
"התחלתי ללמוד קורס חדש וזה ממש מעניין אותי.",
"אני נהנה מהעבודה שלי ומרגיש שאני לומד הרבה.",
"אני מחכה לפגוש את החברים שלי במסיבה בסוף השבוע.",
"הבוקר התחלתי ריצה והרגשתי מלאת אנרגיה.",
"יש לי תוכניות גדולות לעתיד ואני מתכוון להשיג אותן.",
"אני אוהב לבלות עם הכלב שלי בפארק אחרי העבודה.",
"היה לי יום מוצלח בעבודה ואני מרוצה מההישגים שלי.",
"אני מנסה מתכון חדש לארוחת ערב ואני מתרגש איך יצא.",
"היה כיף לראות את הסדרה האהובה עלי בטלוויזיה.",
"אני עובד על פרויקט חדש בעבודה ומרגיש שהוא מתקדם טוב.",
"אני מתכננת לצאת לטיול אופניים בסוף השבוע.",
"נהניתי משיחה טובה עם חבר ותיק היום.",
"הזמנתי כרטיסים למופע שאני ממש מחכה לו.",
"אני אוהבת את התקופה הזו של השנה, הכל פורח ויפה.",
"אני מרגיש קצת מדוכא לאחרונה, אבל אני מנסה להתמודד.",
"לפעמים אני תוהה אם יש בכלל משמעות לכל זה.",
"אני עייפה מכל המאבקים האלה, אבל אני עדיין כאן.",
"אני מרגיש שכלום לא מצליח לי, וזה מתסכל.",
"אני עייפה מהעבודה, לפעמים אני רק רוצה לברוח.",
"יש ימים שאני פשוט לא מצליח למצוא את המוטיבציה.",
"אני מרגישה שאין לי מספיק כוחות להתמודד עם הכל.",
"אני מתחיל לאבד את התקווה, אבל מנסה להחזיק מעמד.",
"אני מרגיש שאני לא מתקדם לשום מקום.",
"יש לי הרבה מחשבות רעות, ואני מתקשה להפסיק אותן.",
"החיים לפעמים מרגישים כמו מאבק בלתי נגמר.",
"אני מרגישה שמשהו חסר, אבל לא יודעת מה.",
"אני מוצא את עצמי מתמודד עם הרבה קשיים לאחרונה.",
"יש לי תחושה שאני לא באמת שייך לשום מקום.",
"אני עייפה מלהתמודד עם כל הבעיות שלי לבד.",
"אני מרגיש שאין לי הרבה למה לצפות.",
"אני מרגישה שאני לא עושה מספיק עם החיים שלי.",
"אני מנסה למצוא פתרון, אבל הכל נראה לי סתום.",
"אני מתחיל לשאול את עצמי אם משהו ישתנה אי פעם.",
"לפעמים אני פשוט רוצה להיעלם לזמן מה.",
"אני לא בטוח כמה זמן אוכל להמשיך ככה.",
"אני מרגישה לבד בעולם הזה, אף אחד לא באמת מבין אותי.",
"כל הזמן יש לי מחשבות רעות, ואני לא מצליח להפסיק אותן.",
"אני פשוט רוצה שהכאב הזה יפסיק, אני לא יודעת איך.",
"אני מרגיש שאני נמצא בחושך, ואין לי מושג איך לצאת ממנו.",
"אני מתחיל לחשוב שאולי אין לזה תקווה.",
"אני מרגישה שאני נאבקת ללא הפסקה ולא רואה תוצאות.",
"אני מרגיש שכל מה שאני עושה הוא חסר משמעות.",
"אני מרגישה שהחיים פשוט עוברים לידי.",
"אני חושב על כל הפעמים שנכשלתי ולא מצליח לשכוח.",
"אני לא מצליחה למצוא סיבה לחייך יותר.",
"אני מרגיש שהכאב הזה לא ייגמר אף פעם.",
"אני חושבת שאני פשוט לא מתאימה לעולם הזה.",
"אני מרגיש שהכל רק הולך ונהיה יותר גרוע.",
"אני לא יודעת אם יש לי מספיק כוח להמשיך הלאה.",
"אני לא רואה את האור בקצה המנהרה.",
"אני מתחיל להרגיש שאין לי מקום בעולם הזה.",
"אני מרגישה שאני נופלת לתוך בור עמוק ולא מצליחה לצאת.",
"אני מתוסכל מהמצב שלי ומרגיש חסר אונים.",
"אני מרגישה שאני הולכת לאיבוד בתוך עצמי.",
"אני חושב על לסיים את הכל, זה נראה לי הפתרון היחיד.",
"אני כבר לא רואה סיבה להמשיך לחיות, הכל חסר טעם.",
"אני לא יודעת אם אצליח לעבור עוד יום, זה כבד מדי.",
"אני מרגיש שאין לי עוד מה להפסיד, אולי עדיף פשוט להיעלם.",
"אני עייפה מהחיים האלה, אני רק רוצה שהכל ייגמר.",
"אני מרגיש שהמחשבות האלה משתלטות עלי ואני לא מצליח לעצור אותן.",
"אני לא רואה דרך אחרת לצאת מהמצב הזה.",
"אני חושבת שזהו, אני לא יכולה להתמודד יותר.",
"אני מרגיש שהעולם יהיה מקום טוב יותר בלעדיי.",
"אני לא יודעת אם אצליח לשרוד את זה לבד.",
"אני מתחיל להרגיש שאין לי עוד ברירה.",
"אני חושב שהסוף הוא הפתרון היחיד שנשאר לי.",
"אני עייפה מלנסות להיאבק כל יום מחדש.",
"אני מרגיש שהכל פשוט חסר טעם.",
"אני לא רואה שום תקווה לשינוי במצב שלי.",
"אני חושבת שהעולם יהיה טוב יותר בלעדיי.",
"אני לא יודע כמה זמן עוד אוכל להחזיק מעמד.",
"אני מרגישה שאני על סף שבירה.",
"אני חושב שעדיף לי לסיים את הכל עכשיו.",
"אני כבר לא רואה סיבה להמשיך.",
"אני עומד לסיים עם זה הלילה, כבר החלטתי.",
"אין טעם לחכות יותר, הלילה אני מסיים עם הכל.",
"אני לא יכולה להמשיך יותר, זה נגמר עבורי.",
"אני הולך לעשות את זה עכשיו, לא משנה מה יקרה.",
"הלילה אני מסיימת את הכל, זה הסוף.",
"אני מתכוון לעשות את זה הלילה, זה הפתרון היחיד.",
"הכל יהיה נגמר בקרוב, אני בטוח בזה.",
"אני יודעת שזה הסוף, אני לא יכולה להילחם יותר.",
"אני הולך לסיים את זה עוד היום, אין טעם לחכות.",
"היום אני שם קץ לכל הסבל הזה.",
"אני לא יכול להמשיך לחיות ככה, זה נגמר.",
"החלטתי שהלילה זה יסתיים, אין דרך חזרה.",
"אני הולכת לסיים את זה עכשיו, אין לי ברירה אחרת.",
"אני מתכוון לשים לזה סוף, אני לא יכול להמשיך.",
"אני לא רואה שום סיבה לחכות יותר.",
"זה הלילה האחרון שלי, אני יודעת את זה.",
"אין לי עוד כוח להמשיך, זה נגמר.",
"אני לא מסוגל יותר לחיות, זה חייב להיגמר.",
"אני לא רואה עוד פתרון מלבד לסיים את זה.",
"אני הולכת לסיים את זה הלילה, זה הפתרון היחיד שנשאר לי.",
"אני עומד על הגג והולך לקפוץ למטה",
"אני מאבד את ההכרה",
"אני עם זכוכית ביד מתכנן לחתוך בצוואר",
"חתכתי את עצמי יורד מלא דם",
"החבל כרוך על צווארי הולך לקפוץ מהכסא",
"הלכתי לאיבוד עם עצמי חושב על המשך הדרך",
"אני בקומה 50 חושב לקפוץ",
"עשיתי שורות קוק במשך שבועיים וניסיתי לגמור עם זה כבר",
"תן לי מתכון לרעל"
]

In [14]:
bert_model.eval()  # Set to eval mode

# 2. Tokenize your messages
inputs = tokenizer(messages, padding=True, truncation=True, max_length=512, return_tensors="pt")

# 3. Send tensors to device (CPU/GPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
bert_model.to(device)
inputs = {k: v.to(device) for k, v in inputs.items()}

# 4. Get predictions
with torch.no_grad():
    outputs = bert_model(**inputs)

# 5. Interpret predictions
logits = outputs.logits
probs = torch.softmax(logits, dim=1)

# If it's binary classification
predictions = torch.argmax(probs, dim=1)

# Print results
for message, prob, pred in zip(messages, probs, predictions):
    #print(f"({message}, {pred.item()} , {prob[1].item():.4f})")
    print(f"{prob[1].item():.4f}")

Message: היום היה יום נחמד, יצאתי לטיול בפארק.
Probability of class 1: 0.0004, Predicted class: 0
----------------------------------------
Message: אני מתכנן לצאת לארוחת ערב עם חברים בסוף השבוע.
Probability of class 1: 0.0004, Predicted class: 0
----------------------------------------
Message: נהנתי מאוד לראות את הסרט החדש בקולנוע.
Probability of class 1: 0.0004, Predicted class: 0
----------------------------------------
Message: אני אוהבת לבלות עם המשפחה שלי בסופי שבוע.
Probability of class 1: 0.0004, Predicted class: 0
----------------------------------------
Message: הלכתי לחדר כושר והרגשתי מעולה אחרי האימון.
Probability of class 1: 0.0004, Predicted class: 0
----------------------------------------
Message: אני מתרגש לטוס לחופשה בקרוב.
Probability of class 1: 0.0004, Predicted class: 0
----------------------------------------
Message: התחלתי ללמוד קורס חדש וזה ממש מעניין אותי.
Probability of class 1: 0.0004, Predicted class: 0
----------------------------------------
Message: אני